# Baptista et al. §5.4 config, applied to Matérn fields, $N=2$

The matched control for `baptista_rectangles_n2_reproduction.ipynb`: **their pipeline, verbatim,
with our two multiband Matérn fields substituted for their two binary rectangles.** Nothing else
is changed except what the data change forces.

The question: Baptista, Dasgupta, Kovachki, Oberai & Stuart
([arXiv:2501.15785](https://arxiv.org/abs/2501.15785), §5.4) show that a U-Net trained on two
binary rectangles collapses onto the training set. Does the same pipeline do the same thing on
two Gaussian random fields?

**This is not an arm of the multiband study.** None of the repo's locked conventions apply —
not $\sigma_{max}=10$, not the per-band ring metric, not `exclude_nn`. This is their sampler,
their architecture, their optimizer, their protocol. Its numbers are comparable to the
rectangles notebook and to nothing else in the repo.

## What is inherited unchanged

Every code cell marked *verbatim* below is byte-identical to the corresponding cell of
`baptista_rectangles_n2_reproduction.ipynb`, which transcribes
[`baptistar/DiffusionModelDynamics`](https://github.com/baptistar/DiffusionModelDynamics)
`RectangleImages/`: `training/networks.py` (`SongUNet`, `EDMPrecond`), `training/loss.py`
(`EDMLoss`) and `generate.py` (`edm_sampler`). The training loop transcribes `main.py:73-107`.

Config, all from `main.py` unless noted:

| | setting | source |
|---|---|---|
| optimizer | Adam, `lr=10e-4`, `betas=[0.9,0.999]`, `eps=1e-8` | `main.py:61` |
| lr schedule | `lr = 10e-4 * min(cur_nimg / 1e7, 1)` — never exceeds $10^{-5}$ here | `main.py:19,89` |
| EMA | `ema_halflife_kimg=500`, `ema_rampup_ratio=0.05`; samples drawn from EMA | `main.py:20-21,96-102,131` |
| batching | `DataLoader(batch_size=1, shuffle=True)` — 2 updates per epoch, not one batch of 2 | `main.py:37` |
| duration | `epochs = 50000` ⇒ 100,000 optimizer updates | `main.py:15` |
| loss | `EDMLoss(P_mean=-1.2, P_std=1.2, sigma_data=0.5)`, reduced `loss.sum()/B` | `loss.py:67`, `main.py:84` |
| $\sigma_{data}$ | **0.5**, the EDM default — *not* estimated from the data | `networks.py:640` |
| grad guard | `nan_to_num(grad, nan=0, posinf=1e5, neginf=-1e5)` | `main.py:91-93` |
| sampler | `edm_sampler(ema, latents, num_steps=40)`, everything else defaulted: $\sigma\in[0.002,80]$, $\rho=7$, `S_churn=0` ⇒ deterministic Heun (ODE) | `main.py:134`, `generate.py:27-28` |
| eval protocol | 100 samples every 1000 optimization steps | paper §5.4 prose |

## What the data change forces

Four changes, and no others.

**1. `img_resolution` 64 → 128.** Our fields are $128\times128$.

**2. `attn_resolutions` `[16]` → `[32]`.** `SongUNet` computes `res = img_resolution >> level`
(`networks.py:281`), so with `channel_mult=[2,2,2]` the three levels sit at 128 / 64 / 32 rather
than 64 / 32 / 16. A literal `[16]` would therefore never fire and would silently delete the
encoder attention. `[32]` puts attention on the same *level* — the bottleneck — and reproduces
**all six Figure-18 parameter counts exactly** (asserted below). Literal `[16]` would give
55,497 / 217,105 / 858,657 / 3,415,105 instead of 57,017 / 222,705 / 880,097 / 3,498,945.

**3. The collapse metric.** Theirs binarizes at 0.5 and counts exact pixel matches — undefined
for continuous fields. Replaced by their *other* released metric, the raw $L^2$ distance to the
data manifold that `main.py:150-159` logs and `post_process.py` plots as the y-axis of
Figure 17. Details in the metric cell.

**4. Geometry.** See below — this one is a real decision, not a mechanical consequence.

## The geometry arms

Their $\sigma_{max}=80$ runs against $D_- = \|x_0-x_1\| = \sqrt{144+196} = 18.44$, so
$\sigma_{max}/D_- = 4.34$. That ratio is what decides whether the reverse process can confuse the
two modes at all: with $x = y_1 + \sigma\varepsilon$, the GMM weight ratio has exponent
$-D_-^2/(2\sigma^2) - \langle\varepsilon,\Delta\rangle/\sigma$ whose random part has standard
deviation $D_-/\sigma$, so the two components merge at $\sigma \sim D_-$ and separate below it.

Our normalized field pair has $D_- = 185.88$, so a literal $\sigma_{max}=80$ gives **0.43** — the
sampler would start *below* the mixing scale, an order of magnitude off their regime. There are
two ways back into it, and **this notebook runs both by default**:

* **`rescaled`** — multiply the field pair by $18.44/185.88 = 0.0992$ so $D_-$ matches theirs
  exactly, and keep $\sigma_{max}=80$ literal. Matches $D_-$, the sampler ratio, *and* the
  training-$\sigma$/$D_-$ ratio. The individual norms land at 11.69 and 14.84 against their
  12.0 and 14.0, so the pair sits in their geometry on both counts.
* **`sigmamax`** — leave the fields exactly as generated and raise $\sigma_{max}$ to
  $4.34 \times 185.88 = 806.4$. Matches the sampler ratio only.

They are **not** equivalent. Rescaling by $s$ at fixed $\sigma_{max}$ is dynamically the same as
fixed data at $\sigma_{max}/s$ *only if $\sigma_{data}$ scales too* — and $\sigma_{data}$ is pinned
at 0.5 in both. `P_mean`/`P_std` likewise fix the **training** $\sigma$ distribution in absolute
terms, so on `sigmamax` the network still never sees noise anywhere near $D_-$ during training even
though the sampler starts above it. Running both separates the sampler axis from the data-scale
axis; agreement between them would be evidence that neither is load-bearing.

A third arm, **`unit`** (data as generated, $\sigma_{max}=80$, ratio 0.43), is the literal
unmodified control. It is **off by default** — enable with `FIELD_GEOM='rescaled sigmamax unit'`.

One thing you cannot have: their $D_-$ and their per-pixel std at the same time. For both datasets
$D_- \approx \sqrt{2d}\,\mathrm{std}$, and $d = 16384$ here against their $4096$, so pinning
$D_- = 18.44$ forces std $= 0.104$, i.e. $\sigma_{data}/\mathrm{std} = 4.8$ against their 2.5.
`rescaled` pins $D_-$, because $D_-$ is what sets the mode-mixing scale.

## Two things not to misread

* `main.py` itself uses `iters_plotting=100` and `Neval_samps=16`. The "100 samples every 1000
  optimization steps" of Figure 18 is paper prose only. This notebook takes the prose values, the
  same choice the rectangles notebook made, so the two runs are comparable **to each other**.
* `loss_fn(net, x).sum() / x.size(0)` (`main.py:84`) sums over pixels. At $128^2$ that is
  $4\times$ the gradient magnitude of $64^2$ at the same learning rate. Kept literal. It shifts
  memorization *timing* relative to the rectangles run and must not be read as a data effect.

## Setup

`src/` is used for `device_utils`, `multiband_data_utils` (the dataset generator, shared with
every other field experiment) and a numerical cross-check against `src/edm.py`. **No file in
`src/` is modified or needed to run this notebook**, so every previously committed result is
untouched.

In [ ]:
import os, sys, math, time, copy, json
import numpy as np
import torch
import matplotlib

# Headless-safe: picks Agg under SLURM (no $DISPLAY), leaves inline alone in Jupyter.
if not os.environ.get('DISPLAY') and not hasattr(sys, 'ps1'):
    matplotlib.use('Agg')
import matplotlib.pyplot as plt

# -- path setup (same pattern as the other notebooks in this folder) --
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
src_dir = os.path.join(repo_root, 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from device_utils import resolve_device

DEVICE = resolve_device()           # cuda > mps > cpu
torch.backends.cudnn.benchmark = True

N_TRAIN = 2      # main.py:13
GRID    = 128    # our fields (main.py's rectangles are 64)

# Heavy artifacts (resume state is ~40 MB for the 3.5M arm) can be redirected off a shared
# project quota:   export FIELD_RESULTS_DIR=$SCRATCH/matern_n2
results_dir = os.environ.get('FIELD_RESULTS_DIR',
                             os.path.join(repo_root, 'results', 'data'))
fig_dir = os.environ.get('FIELD_FIG_DIR',
                         os.path.join(repo_root, 'results', 'figures'))
os.makedirs(results_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)

print(f'torch {torch.__version__}')
print(f'results -> {results_dir}')
print(f'figures -> {fig_dir}')
print(f'device: {DEVICE}')
if DEVICE.type == 'cuda':
    p = torch.cuda.get_device_properties(0)
    print(f'gpu: {p.name}, {p.total_memory / 1e9:.1f} GB, cc {p.major}.{p.minor}')
else:
    print('WARNING: this notebook is sized for a CUDA GPU. 100k updates at 128x128 is not '
          'practical on CPU/MPS -- raise SMOKE if you are just checking the pipeline runs.')

## The EDM network — verbatim

Copied byte-for-byte from `baptista_rectangles_n2_reproduction.ipynb`, which transcribes
`RectangleImages/training/networks.py` from the paper's repo (byte-identical to
[NVlabs/edm](https://github.com/NVlabs/edm) `training/networks.py` apart from comments).

Only two mechanical changes, neither touching any computation: the
`@persistence.persistent_class` decorators and the `torch_utils` import are dropped (pickle
plumbing this notebook does not use), and `DhariwalUNet` / `VPPrecond` / `VEPrecond` /
`iDDPMPrecond` are omitted (`main.py` uses `SongUNet` under `EDMPrecond` and nothing else).

> Karras, Aittala, Aila & Laine, *Elucidating the Design Space of Diffusion-Based Generative
> Models*, NeurIPS 2022. Code © 2022 NVIDIA CORPORATION & AFFILIATES, released under
> CC BY-NC-SA 4.0. Reproduced here for research use.

In [ ]:
# ---------------------------------------------------------------------------------------------
# Transcribed verbatim from RectangleImages/training/networks.py in baptistar/DiffusionModelDynamics
# (byte-identical to NVlabs/edm training/networks.py apart from comments).
#
# Copyright (c) 2022, NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# Licensed under CC BY-NC-SA 4.0 -- http://creativecommons.org/licenses/by-nc-sa/4.0/
# "Elucidating the Design Space of Diffusion-Based Generative Models", Karras et al., NeurIPS 2022.
#
# Changes: @persistence decorators and the torch_utils import removed (pickle plumbing only);
# DhariwalUNet / VPPrecond / VEPrecond / iDDPMPrecond omitted (unused by main.py).
# No computational change.
# ---------------------------------------------------------------------------------------------

from torch.nn.functional import silu

def weight_init(shape, mode, fan_in, fan_out):
    if mode == 'xavier_uniform': return np.sqrt(6 / (fan_in + fan_out)) * (torch.rand(*shape) * 2 - 1)
    if mode == 'xavier_normal':  return np.sqrt(2 / (fan_in + fan_out)) * torch.randn(*shape)
    if mode == 'kaiming_uniform': return np.sqrt(3 / fan_in) * (torch.rand(*shape) * 2 - 1)
    if mode == 'kaiming_normal':  return np.sqrt(1 / fan_in) * torch.randn(*shape)
    raise ValueError(f'Invalid init mode "{mode}"')

#----------------------------------------------------------------------------
# Fully-connected layer.

class Linear(torch.nn.Module):
    def __init__(self, in_features, out_features, bias=True, init_mode='kaiming_normal', init_weight=1, init_bias=0):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        init_kwargs = dict(mode=init_mode, fan_in=in_features, fan_out=out_features)
        self.weight = torch.nn.Parameter(weight_init([out_features, in_features], **init_kwargs) * init_weight)
        self.bias = torch.nn.Parameter(weight_init([out_features], **init_kwargs) * init_bias) if bias else None

    def forward(self, x):
        x = x @ self.weight.to(x.dtype).t()
        if self.bias is not None:
            x = x.add_(self.bias.to(x.dtype))
        return x

#----------------------------------------------------------------------------
# Convolutional layer with optional up/downsampling.

class Conv2d(torch.nn.Module):
    def __init__(self,
        in_channels, out_channels, kernel, bias=True, up=False, down=False,
        resample_filter=[1,1], fused_resample=False, init_mode='kaiming_normal', init_weight=1, init_bias=0,
    ):
        assert not (up and down)
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.up = up
        self.down = down
        self.fused_resample = fused_resample
        init_kwargs = dict(mode=init_mode, fan_in=in_channels*kernel*kernel, fan_out=out_channels*kernel*kernel)
        self.weight = torch.nn.Parameter(weight_init([out_channels, in_channels, kernel, kernel], **init_kwargs) * init_weight) if kernel else None
        self.bias = torch.nn.Parameter(weight_init([out_channels], **init_kwargs) * init_bias) if kernel and bias else None
        f = torch.as_tensor(resample_filter, dtype=torch.float32)
        f = f.ger(f).unsqueeze(0).unsqueeze(1) / f.sum().square()
        self.register_buffer('resample_filter', f if up or down else None)

    def forward(self, x):
        w = self.weight.to(x.dtype) if self.weight is not None else None
        b = self.bias.to(x.dtype) if self.bias is not None else None
        f = self.resample_filter.to(x.dtype) if self.resample_filter is not None else None
        w_pad = w.shape[-1] // 2 if w is not None else 0
        f_pad = (f.shape[-1] - 1) // 2 if f is not None else 0

        if self.fused_resample and self.up and w is not None:
            x = torch.nn.functional.conv_transpose2d(x, f.mul(4).tile([self.in_channels, 1, 1, 1]), groups=self.in_channels, stride=2, padding=max(f_pad - w_pad, 0))
            x = torch.nn.functional.conv2d(x, w, padding=max(w_pad - f_pad, 0))
        elif self.fused_resample and self.down and w is not None:
            x = torch.nn.functional.conv2d(x, w, padding=w_pad+f_pad)
            x = torch.nn.functional.conv2d(x, f.tile([self.out_channels, 1, 1, 1]), groups=self.out_channels, stride=2)
        else:
            if self.up:
                x = torch.nn.functional.conv_transpose2d(x, f.mul(4).tile([self.in_channels, 1, 1, 1]), groups=self.in_channels, stride=2, padding=f_pad)
            if self.down:
                x = torch.nn.functional.conv2d(x, f.tile([self.in_channels, 1, 1, 1]), groups=self.in_channels, stride=2, padding=f_pad)
            if w is not None:
                x = torch.nn.functional.conv2d(x, w, padding=w_pad)
        if b is not None:
            x = x.add_(b.reshape(1, -1, 1, 1))
        return x

#----------------------------------------------------------------------------
# Group normalization.

class GroupNorm(torch.nn.Module):
    def __init__(self, num_channels, num_groups=32, min_channels_per_group=4, eps=1e-5):
        super().__init__()
        self.num_groups = min(num_groups, num_channels // min_channels_per_group)
        self.eps = eps
        self.weight = torch.nn.Parameter(torch.ones(num_channels))
        self.bias = torch.nn.Parameter(torch.zeros(num_channels))

    def forward(self, x):
        x = torch.nn.functional.group_norm(x, num_groups=self.num_groups, weight=self.weight.to(x.dtype), bias=self.bias.to(x.dtype), eps=self.eps)
        return x

#----------------------------------------------------------------------------
# Attention weight computation, i.e., softmax(Q^T * K).
# Performs all computation using FP32, but uses the original datatype for
# inputs/outputs/gradients to conserve memory.

class AttentionOp(torch.autograd.Function):
    @staticmethod
    def forward(ctx, q, k):
        w = torch.einsum('ncq,nck->nqk', q.to(torch.float32), (k / np.sqrt(k.shape[1])).to(torch.float32)).softmax(dim=2).to(q.dtype)
        ctx.save_for_backward(q, k, w)
        return w

    @staticmethod
    def backward(ctx, dw):
        q, k, w = ctx.saved_tensors
        db = torch._softmax_backward_data(grad_output=dw.to(torch.float32), output=w.to(torch.float32), dim=2, input_dtype=torch.float32)
        dq = torch.einsum('nck,nqk->ncq', k.to(torch.float32), db).to(q.dtype) / np.sqrt(k.shape[1])
        dk = torch.einsum('ncq,nqk->nck', q.to(torch.float32), db).to(k.dtype) / np.sqrt(k.shape[1])
        return dq, dk

#----------------------------------------------------------------------------
# Unified U-Net block with optional up/downsampling and self-attention.
# Represents the union of all features employed by the DDPM++, NCSN++, and
# ADM architectures.

class UNetBlock(torch.nn.Module):
    def __init__(self,
        in_channels, out_channels, emb_channels, up=False, down=False, attention=False,
        num_heads=None, channels_per_head=64, dropout=0, skip_scale=1, eps=1e-5,
        resample_filter=[1,1], resample_proj=False, adaptive_scale=True,
        init=dict(), init_zero=dict(init_weight=0), init_attn=None,
    ):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.emb_channels = emb_channels
        self.num_heads = 0 if not attention else num_heads if num_heads is not None else out_channels // channels_per_head
        self.dropout = dropout
        self.skip_scale = skip_scale
        self.adaptive_scale = adaptive_scale

        self.norm0 = GroupNorm(num_channels=in_channels, eps=eps)
        self.conv0 = Conv2d(in_channels=in_channels, out_channels=out_channels, kernel=3, up=up, down=down, resample_filter=resample_filter, **init)
        self.affine = Linear(in_features=emb_channels, out_features=out_channels*(2 if adaptive_scale else 1), **init)
        self.norm1 = GroupNorm(num_channels=out_channels, eps=eps)
        self.conv1 = Conv2d(in_channels=out_channels, out_channels=out_channels, kernel=3, **init_zero)

        self.skip = None
        if out_channels != in_channels or up or down:
            kernel = 1 if resample_proj or out_channels!= in_channels else 0
            self.skip = Conv2d(in_channels=in_channels, out_channels=out_channels, kernel=kernel, up=up, down=down, resample_filter=resample_filter, **init)

        if self.num_heads:
            self.norm2 = GroupNorm(num_channels=out_channels, eps=eps)
            self.qkv = Conv2d(in_channels=out_channels, out_channels=out_channels*3, kernel=1, **(init_attn if init_attn is not None else init))
            self.proj = Conv2d(in_channels=out_channels, out_channels=out_channels, kernel=1, **init_zero)

    def forward(self, x, emb):
        orig = x
        x = self.conv0(silu(self.norm0(x)))

        params = self.affine(emb).unsqueeze(2).unsqueeze(3).to(x.dtype)
        if self.adaptive_scale:
            scale, shift = params.chunk(chunks=2, dim=1)
            x = silu(torch.addcmul(shift, self.norm1(x), scale + 1))
        else:
            x = silu(self.norm1(x.add_(params)))

        x = self.conv1(torch.nn.functional.dropout(x, p=self.dropout, training=self.training))
        x = x.add_(self.skip(orig) if self.skip is not None else orig)
        x = x * self.skip_scale

        if self.num_heads:
            q, k, v = self.qkv(self.norm2(x)).reshape(x.shape[0] * self.num_heads, x.shape[1] // self.num_heads, 3, -1).unbind(2)
            w = AttentionOp.apply(q, k)
            a = torch.einsum('nqk,nck->ncq', w, v)
            x = self.proj(a.reshape(*x.shape)).add_(x)
            x = x * self.skip_scale
        return x

#----------------------------------------------------------------------------
# Timestep embedding used in the DDPM++ and ADM architectures.

class PositionalEmbedding(torch.nn.Module):
    def __init__(self, num_channels, max_positions=10000, endpoint=False):
        super().__init__()
        self.num_channels = num_channels
        self.max_positions = max_positions
        self.endpoint = endpoint

    def forward(self, x):
        freqs = torch.arange(start=0, end=self.num_channels//2, dtype=torch.float32, device=x.device)
        freqs = freqs / (self.num_channels // 2 - (1 if self.endpoint else 0))
        freqs = (1 / self.max_positions) ** freqs
        x = x.ger(freqs.to(x.dtype))
        x = torch.cat([x.cos(), x.sin()], dim=1)
        return x

#----------------------------------------------------------------------------
# Timestep embedding used in the NCSN++ architecture.

class FourierEmbedding(torch.nn.Module):
    def __init__(self, num_channels, scale=16):
        super().__init__()
        self.register_buffer('freqs', torch.randn(num_channels // 2) * scale)

    def forward(self, x):
        x = x.ger((2 * np.pi * self.freqs).to(x.dtype))
        x = torch.cat([x.cos(), x.sin()], dim=1)
        return x

#----------------------------------------------------------------------------
# Reimplementation of the DDPM++ and NCSN++ architectures from the paper
# "Score-Based Generative Modeling through Stochastic Differential
# Equations". Equivalent to the original implementation by Song et al.,
# available at https://github.com/yang-song/score_sde_pytorch

class SongUNet(torch.nn.Module):
    def __init__(self,
        img_resolution,                     # Image resolution at input/output.
        in_channels,                        # Number of color channels at input.
        out_channels,                       # Number of color channels at output.
        label_dim           = 0,            # Number of class labels, 0 = unconditional.
        augment_dim         = 0,            # Augmentation label dimensionality, 0 = no augmentation.

        model_channels      = 128,          # Base multiplier for the number of channels.
        channel_mult        = [1,2,2,2],    # Per-resolution multipliers for the number of channels.
        channel_mult_emb    = 4,            # Multiplier for the dimensionality of the embedding vector.
        num_blocks          = 4,            # Number of residual blocks per resolution.
        attn_resolutions    = [16],         # List of resolutions with self-attention.
        dropout             = 0.10,         # Dropout probability of intermediate activations.
        label_dropout       = 0,            # Dropout probability of class labels for classifier-free guidance.

        embedding_type      = 'positional', # Timestep embedding type: 'positional' for DDPM++, 'fourier' for NCSN++.
        channel_mult_noise  = 1,            # Timestep embedding size: 1 for DDPM++, 2 for NCSN++.
        encoder_type        = 'standard',   # Encoder architecture: 'standard' for DDPM++, 'residual' for NCSN++.
        decoder_type        = 'standard',   # Decoder architecture: 'standard' for both DDPM++ and NCSN++.
        resample_filter     = [1,1],        # Resampling filter: [1,1] for DDPM++, [1,3,3,1] for NCSN++.
    ):
        assert embedding_type in ['fourier', 'positional']
        assert encoder_type in ['standard', 'skip', 'residual']
        assert decoder_type in ['standard', 'skip']

        super().__init__()
        self.label_dropout = label_dropout
        emb_channels = model_channels * channel_mult_emb
        noise_channels = model_channels * channel_mult_noise
        init = dict(init_mode='xavier_uniform')
        init_zero = dict(init_mode='xavier_uniform', init_weight=1e-5)
        init_attn = dict(init_mode='xavier_uniform', init_weight=np.sqrt(0.2))
        block_kwargs = dict(
            emb_channels=emb_channels, num_heads=1, dropout=dropout, skip_scale=np.sqrt(0.5), eps=1e-6,
            resample_filter=resample_filter, resample_proj=True, adaptive_scale=False,
            init=init, init_zero=init_zero, init_attn=init_attn,
        )

        # Mapping.
        self.map_noise = PositionalEmbedding(num_channels=noise_channels, endpoint=True) if embedding_type == 'positional' else FourierEmbedding(num_channels=noise_channels)
        self.map_label = Linear(in_features=label_dim, out_features=noise_channels, **init) if label_dim else None
        self.map_augment = Linear(in_features=augment_dim, out_features=noise_channels, bias=False, **init) if augment_dim else None
        self.map_layer0 = Linear(in_features=noise_channels, out_features=emb_channels, **init)
        self.map_layer1 = Linear(in_features=emb_channels, out_features=emb_channels, **init)

        # Encoder.
        self.enc = torch.nn.ModuleDict()
        cout = in_channels
        caux = in_channels
        for level, mult in enumerate(channel_mult):
            res = img_resolution >> level
            if level == 0:
                cin = cout
                cout = model_channels
                self.enc[f'{res}x{res}_conv'] = Conv2d(in_channels=cin, out_channels=cout, kernel=3, **init)
            else:
                self.enc[f'{res}x{res}_down'] = UNetBlock(in_channels=cout, out_channels=cout, down=True, **block_kwargs)
                if encoder_type == 'skip':
                    self.enc[f'{res}x{res}_aux_down'] = Conv2d(in_channels=caux, out_channels=caux, kernel=0, down=True, resample_filter=resample_filter)
                    self.enc[f'{res}x{res}_aux_skip'] = Conv2d(in_channels=caux, out_channels=cout, kernel=1, **init)
                if encoder_type == 'residual':
                    self.enc[f'{res}x{res}_aux_residual'] = Conv2d(in_channels=caux, out_channels=cout, kernel=3, down=True, resample_filter=resample_filter, fused_resample=True, **init)
                    caux = cout
            for idx in range(num_blocks):
                cin = cout
                cout = model_channels * mult
                attn = (res in attn_resolutions)
                self.enc[f'{res}x{res}_block{idx}'] = UNetBlock(in_channels=cin, out_channels=cout, attention=attn, **block_kwargs)
        skips = [block.out_channels for name, block in self.enc.items() if 'aux' not in name]

        # Decoder.
        self.dec = torch.nn.ModuleDict()
        for level, mult in reversed(list(enumerate(channel_mult))):
            res = img_resolution >> level
            if level == len(channel_mult) - 1:
                self.dec[f'{res}x{res}_in0'] = UNetBlock(in_channels=cout, out_channels=cout, attention=True, **block_kwargs)
                self.dec[f'{res}x{res}_in1'] = UNetBlock(in_channels=cout, out_channels=cout, **block_kwargs)
            else:
                self.dec[f'{res}x{res}_up'] = UNetBlock(in_channels=cout, out_channels=cout, up=True, **block_kwargs)
            for idx in range(num_blocks + 1):
                cin = cout + skips.pop()
                cout = model_channels * mult
                attn = (idx == num_blocks and res in attn_resolutions)
                self.dec[f'{res}x{res}_block{idx}'] = UNetBlock(in_channels=cin, out_channels=cout, attention=attn, **block_kwargs)
            if decoder_type == 'skip' or level == 0:
                if decoder_type == 'skip' and level < len(channel_mult) - 1:
                    self.dec[f'{res}x{res}_aux_up'] = Conv2d(in_channels=out_channels, out_channels=out_channels, kernel=0, up=True, resample_filter=resample_filter)
                self.dec[f'{res}x{res}_aux_norm'] = GroupNorm(num_channels=cout, eps=1e-6)
                self.dec[f'{res}x{res}_aux_conv'] = Conv2d(in_channels=cout, out_channels=out_channels, kernel=3, **init_zero)

    def forward(self, x, noise_labels, class_labels, augment_labels=None):
        # Mapping.
        emb = self.map_noise(noise_labels)
        emb = emb.reshape(emb.shape[0], 2, -1).flip(1).reshape(*emb.shape) # swap sin/cos
        if self.map_label is not None:
            tmp = class_labels
            if self.training and self.label_dropout:
                tmp = tmp * (torch.rand([x.shape[0], 1], device=x.device) >= self.label_dropout).to(tmp.dtype)
            emb = emb + self.map_label(tmp * np.sqrt(self.map_label.in_features))
        if self.map_augment is not None and augment_labels is not None:
            emb = emb + self.map_augment(augment_labels)
        emb = silu(self.map_layer0(emb))
        emb = silu(self.map_layer1(emb))

        # Encoder.
        skips = []
        aux = x
        for name, block in self.enc.items():
            if 'aux_down' in name:
                aux = block(aux)
            elif 'aux_skip' in name:
                x = skips[-1] = x + block(aux)
            elif 'aux_residual' in name:
                x = skips[-1] = aux = (x + block(aux)) / np.sqrt(2)
            else:
                x = block(x, emb) if isinstance(block, UNetBlock) else block(x)
                skips.append(x)

        # Decoder.
        aux = None
        tmp = None
        for name, block in self.dec.items():
            if 'aux_up' in name:
                aux = block(aux)
            elif 'aux_norm' in name:
                tmp = block(x)
            elif 'aux_conv' in name:
                tmp = block(silu(tmp))
                aux = tmp if aux is None else tmp + aux
            else:
                if x.shape[1] != block.in_channels:
                    x = torch.cat([x, skips.pop()], dim=1)
                x = block(x, emb)
        return aux

#----------------------------------------------------------------------------
class EDMPrecond(torch.nn.Module):
    def __init__(self,
        img_resolution,                     # Image resolution.
        img_channels,                       # Number of color channels.
        label_dim       = 0,                # Number of class labels, 0 = unconditional.
        use_fp16        = False,            # Execute the underlying model at FP16 precision?
        sigma_min       = 0,                # Minimum supported noise level.
        sigma_max       = float('inf'),     # Maximum supported noise level.
        sigma_data      = 0.5,              # Expected standard deviation of the training data.
        model_type      = 'DhariwalUNet',   # Class name of the underlying model.
        **model_kwargs,                     # Keyword arguments for the underlying model.
    ):
        super().__init__()
        self.img_resolution = img_resolution
        self.img_channels = img_channels
        self.label_dim = label_dim
        self.use_fp16 = use_fp16
        self.sigma_min = sigma_min
        self.sigma_max = sigma_max
        self.sigma_data = sigma_data
        self.model = globals()[model_type](img_resolution=img_resolution, in_channels=img_channels, out_channels=img_channels, label_dim=label_dim, **model_kwargs)

    def forward(self, x, sigma, class_labels=None, force_fp32=False, **model_kwargs):
        x = x.to(torch.float32)
        sigma = sigma.to(torch.float32).reshape(-1, 1, 1, 1)
        class_labels = None if self.label_dim == 0 else torch.zeros([1, self.label_dim], device=x.device) if class_labels is None else class_labels.to(torch.float32).reshape(-1, self.label_dim)
        dtype = torch.float16 if (self.use_fp16 and not force_fp32 and x.device.type == 'cuda') else torch.float32

        c_skip = self.sigma_data ** 2 / (sigma ** 2 + self.sigma_data ** 2) #1
        c_out = sigma * self.sigma_data / (sigma ** 2 + self.sigma_data ** 2).sqrt() #0
        c_in = 1 / (self.sigma_data ** 2 + sigma ** 2).sqrt()
        c_noise = sigma.log() / 4

        F_x = self.model((c_in * x).to(dtype), c_noise.flatten(), class_labels=class_labels, **model_kwargs)
        assert F_x.dtype == dtype
        D_x = c_skip * x + c_out * F_x.to(torch.float32)
        return D_x

    def round_sigma(self, sigma):
        return torch.as_tensor(sigma)

### Verification 1 — the parameter counts still match Figure 18

Figure 18's legend gives 57017 / 222705 / 880097 / 3498945 / 13952897 / 55725825 parameters, from
*"varying the number of channels in the first residual block"*.

`SongUNet`'s parameter count does not depend on `img_resolution` **except** through which blocks
receive attention, and `attn_resolutions` is matched against absolute resolutions rather than
level indices. Setting `attn_resolutions=[32]` at `img_resolution=128` attaches attention to the
same level as `[16]` does at 64, and reproduces all six counts exactly. The assert below is the
guard: if the config ever drifts, this fails loudly rather than producing a quietly different
network.

In [ ]:
# main.py:50-57, with img_resolution and attn_resolutions carrying the data change.
CHANNEL_MULT = [2, 2, 2]
ATTN_RES = [GRID >> (len(CHANNEL_MULT) - 1)]    # deepest level: 128>>2 = 32  (main.py: 64>>2 = 16)
assert ATTN_RES == [32], ATTN_RES

NET_KWARGS = dict(
    img_resolution   = GRID,        # main.py:55 has 64; our fields are 128
    img_channels     = 1,
    label_dim        = 0,
    use_fp16         = False,
    model_type       = 'SongUNet',
    embedding_type   = 'positional',
    encoder_type     = 'standard',
    decoder_type     = 'standard',
    channel_mult_noise = 1,
    resample_filter  = [1, 1],
    channel_mult     = CHANNEL_MULT,
    dropout          = 0.0,
    attn_resolutions = ATTN_RES,    # main.py leaves the SongUNet default [16]; see markdown above
)
# num_blocks=4 is a SongUNet default; main.py does not override it.

def build_net(model_channels, device=None):
    net = EDMPrecond(model_channels=model_channels, **NET_KWARGS)
    return net if device is None else net.to(device)

def count_params(net):
    return sum(p.numel() for p in net.parameters())

PAPER_FIG18_PARAMS = {4: 57017, 8: 222705, 16: 880097,
                      32: 3498945, 64: 13952897, 128: 55725825}

print(f"{'model_channels':>15} {'params @128':>13} {'Fig. 18 legend':>15}   match")
for C, target in PAPER_FIG18_PARAMS.items():
    n = count_params(build_net(C))
    assert n == target, f'model_channels={C}: got {n:,}, Figure 18 says {target:,}'
    print(f'{C:>15} {n:>13,} {target:>15,}   exact')
print('\nall six parameter counts reproduce Figure 18 exactly at img_resolution=128')

### Verification 2 — the transcribed preconditioner agrees with `src/edm.py`

The repo has its own `EDMPrecond` in `src/edm.py`, used by every other experiment. This checks
that the transcribed EDM one and the repo's compute the same thing on the same weights, so the
two families of results share a preconditioner definition. Read-only: nothing in `src/` is
touched.

In [ ]:
import edm as src_edm   # the repo's own EDM implementation, unmodified

@torch.no_grad()
def _crosscheck_precond(model_channels=4, batch=4, seed=0):
    torch.manual_seed(seed)
    ref = build_net(model_channels).eval()          # EDM's EDMPrecond (transcribed above)

    # Wrap the *same* SongUNet instance in src/edm.py's EDMPrecond. src's calls net(x, c_noise)
    # positionally, so a 2-arg shim supplies class_labels=None.
    class _TwoArgShim(torch.nn.Module):
        def __init__(self, m):
            super().__init__(); self.m = m
        def forward(self, x, c_noise):
            return self.m(x, c_noise, class_labels=None)

    mine = src_edm.EDMPrecond(_TwoArgShim(ref.model), sigma_data=ref.sigma_data).eval()

    x = torch.randn(batch, 1, GRID, GRID)
    sigma = torch.tensor([0.01, 0.5, 2.0, 40.0])[:batch]
    a = ref(x, sigma)
    b = mine(x, sigma)
    return a, b

a, b = _crosscheck_precond()
max_abs = (a - b).abs().max().item()
print(f'max |EDM.EDMPrecond - src.edm.EDMPrecond| = {max_abs:.3e}  (output scale {a.abs().max():.3f})')
assert torch.allclose(a, b, rtol=0, atol=1e-6), 'src/edm.py EDMPrecond disagrees with EDM reference'
print('src/edm.py EDMPrecond matches the EDM reference')

# The EDM loss weight is also shared logic; check it against src/edm.py's helper.
_s = torch.tensor([0.01, 0.5, 2.0, 40.0])
_w_ref = (_s ** 2 + 0.5 ** 2) / (_s * 0.5) ** 2
assert torch.allclose(_w_ref, src_edm.edm_loss_weight(_s, 0.5))
print('src/edm.py edm_loss_weight matches the EDM reference')

## The loss — verbatim

`training/loss.py:65-81`. `main.py:63-65` constructs `EDMLoss` with no arguments, so
`P_mean=-1.2`, `P_std=1.2`, `sigma_data=0.5` are all the class defaults.

In [ ]:
class EDMLoss:
    """training/loss.py, verbatim. Returns the per-element loss; the caller reduces it."""
    def __init__(self, P_mean=-1.2, P_std=1.2, sigma_data=0.5):
        self.P_mean = P_mean
        self.P_std = P_std
        self.sigma_data = sigma_data

    def __call__(self, net, images, labels=None, augment_pipe=None):
        rnd_normal = torch.randn([images.shape[0], 1, 1, 1], device=images.device)
        sigma = (rnd_normal * self.P_std + self.P_mean).exp()
        weight = (sigma ** 2 + self.sigma_data ** 2) / (sigma * self.sigma_data) ** 2
        y, augment_labels = augment_pipe(images) if augment_pipe is not None else (images, None)
        n = torch.randn_like(y) * sigma
        D_yn = net(y + n, sigma, labels, augment_labels=augment_labels)
        loss = weight * ((D_yn - y) ** 2)
        return loss

## The sampler — verbatim

`generate.py:25-60`, EDM Algorithm 2. `main.py:134` calls `edm_sampler(ema, latents,
num_steps=40)` and leaves `sigma_min=0.002`, `sigma_max=80`, `rho=7`, `S_churn=0`, `S_min=0`,
`S_max=inf`, `S_noise=1` at their defaults.

`S_churn=0` makes `gamma=0`, hence `t_hat == t_cur` and $\sqrt{t_{hat}^2-t_{cur}^2}=0$: the
injected-noise term vanishes and Algorithm 2 degenerates to **deterministic 2nd-order Heun**,
an ODE. §5.4 of the paper describes an SDE. That discrepancy is in their released material, not
a decision this notebook makes; `CFG['S_churn']` defaults to the released 0 and `CFG_SDE_ALT`
holds Karras's CIFAR-10 stochastic preset for a one-line switch.

In [ ]:
# float64 everywhere, exactly as EDM -- except on MPS, which cannot allocate float64 at all.
SAMPLER_DTYPE = torch.float32 if DEVICE.type == 'mps' else torch.float64
if SAMPLER_DTYPE is torch.float32:
    print('WARNING: MPS cannot do float64; sampling in float32. EDM (and a CUDA run) uses '
          'float64 -- do not report numbers from an MPS run.')


def edm_sampler(
    net, latents, class_labels=None, randn_like=torch.randn_like,
    num_steps=18, sigma_min=0.002, sigma_max=80, rho=7,
    S_churn=0, S_min=0, S_max=float('inf'), S_noise=1,
):
    """generate.py, verbatim (EDM Algorithm 2)."""
    # Adjust noise levels based on what's supported by the network.
    sigma_min = max(sigma_min, net.sigma_min)
    sigma_max = min(sigma_max, net.sigma_max)

    # Time step discretization.
    step_indices = torch.arange(num_steps, dtype=SAMPLER_DTYPE, device=latents.device)  # EDM: torch.float64
    t_steps = (sigma_max ** (1 / rho) + step_indices / (num_steps - 1) * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))) ** rho
    t_steps = torch.cat([net.round_sigma(t_steps), torch.zeros_like(t_steps[:1])]) # t_N = 0

    # Main sampling loop.
    x_next = latents.to(SAMPLER_DTYPE) * t_steps[0]  # EDM: torch.float64
    for i, (t_cur, t_next) in enumerate(zip(t_steps[:-1], t_steps[1:])): # 0, ..., N-1
        x_cur = x_next

        # Increase noise temporarily.
        gamma = min(S_churn / num_steps, np.sqrt(2) - 1) if S_min <= t_cur <= S_max else 0
        t_hat = net.round_sigma(t_cur + gamma * t_cur)
        x_hat = x_cur + (t_hat ** 2 - t_cur ** 2).sqrt() * S_noise * randn_like(x_cur)

        # Euler step.
        denoised = net(x_hat, t_hat, class_labels).to(SAMPLER_DTYPE)  # EDM: torch.float64
        d_cur = (x_hat - denoised) / t_hat
        x_next = x_hat + (t_next - t_hat) * d_cur

        # Apply 2nd order correction.
        if i < num_steps - 1:
            denoised = net(x_next, t_next, class_labels).to(SAMPLER_DTYPE)  # EDM: torch.float64
            d_prime = (x_next - denoised) / t_next
            x_next = x_hat + (t_next - t_hat) * (0.5 * d_cur + 0.5 * d_prime)

    return x_next

## Configuration

Every value is `main.py`'s, except the geometry arms (new), `model_channels_sweep` and
`n_eval_samples`/`eval_every` (Figure 18 protocol, from the prose), and `eval_batch` (memory).

`CFG['geometries']` is a **list**: one execution trains every listed geometry in turn, so a single
job covers both arms. Environment overrides let one SLURM array task own one (geometry, arm) pair:
`FIELD_GEOM=sigmamax FIELD_ARMS=32 sbatch ...`


In [ ]:
# True -> a few-minute end-to-end check of the whole pipeline. Set from the environment so a
# smoke run needs no edit to this file:   FIELD_SMOKE=1 python3 -m nbconvert ...
SMOKE = os.environ.get('FIELD_SMOKE', '0').strip().lower() not in ('0', '', 'false', 'no')

CFG = dict(
    # -- from main.py, verbatim --------------------------------------------------------------
    epochs             = 50_000,   # main.py:15
    batch_mode         = 'main_py',# 'main_py'  : DataLoader(batch_size=1, shuffle=True), 2 updates/epoch
                                   # 'paper_bs2': one update/epoch on a batch of 2 (what the prose says)
    lr                 = 10e-4,    # main.py:61
    betas              = (0.9, 0.999),
    eps                = 1e-8,
    lr_rampup_kimg     = 10_000,   # main.py:19  -> lr never exceeds 1e-5 over this run
    ema_halflife_kimg  = 500,      # main.py:20
    ema_rampup_ratio   = 0.05,     # main.py:21
    P_mean             = -1.2,     # EDMLoss defaults, main.py:63-65
    P_std              = 1.2,
    sigma_data         = 0.5,      # networks.py:640 default; NOT estimated from the data

    # -- sampler: main.py:134 calls edm_sampler(ema, latents, num_steps=40), rest defaulted ----
    num_steps          = 40,
    sigma_min          = 0.002,
    rho                = 7,
    S_churn            = 0.0,      # released default -> deterministic Heun (ODE). See CFG_SDE_ALT.
    S_min              = 0.0,
    S_max              = float('inf'),
    S_noise            = 1.0,
    # sigma_max is deliberately NOT here -- it is per-geometry. See GEOM_SPECS below.

    # -- Figure 18 protocol: from the paper text (script not released) -------------------------
    model_channels_sweep = [4, 8, 16, 32],
    n_eval_samples     = 100,      # "we generate 100 samples from each model"
    eval_every         = 1_000,    # "after every 1000 optimization steps"

    # -- the data change -----------------------------------------------------------------------
    rel_threshold      = 0.3,      # collapse threshold on the RELATIVE distance; secondary metric

    # -- run mechanics (not from the paper) ----------------------------------------------------
    seed               = 0,
    latent_seed        = 42,       # eval latents are seeded per evaluation for reproducibility
    eval_batch         = 25,       # 128^2 in float64; lower this first on an OOM
    save_resume_state  = True,     # net+ema+optimizer dumped each eval so a SLURM timeout resumes
    n_sample_grid      = 16,       # Figure 17 shows 16 samples
)

# Karras et al.'s CIFAR-10 stochastic preset, for the SDE reading of the paper text.
# Apply with: CFG.update(CFG_SDE_ALT)
CFG_SDE_ALT = dict(S_churn=30.0, S_min=0.01, S_max=1.0, S_noise=1.007)

# ---------------------------------------------------------------------------------------------
# The geometry arms. Their run sits at sigma_max / D_- = 80 / 18.44 = 4.34; our pair has
# D_- = 185.88, so a literal sigma_max = 80 gives 0.43. Two ways back into their regime:
#
#   'rescaled'  scale the data so D_- matches theirs; keep sigma_max = 80 literal.
#               Matches D_-, the sampler ratio, AND the training-sigma/D_- ratio.
#   'sigmamax'  leave the data as generated; raise sigma_max so the ratio matches.
#               Matches the sampler ratio only -- D_- stays 10x theirs, and P_mean/P_std keep the
#               TRAINING sigma range where it was, so the network still never sees noise near D_-.
#   'unit'      literal control: data as generated, sigma_max = 80, ratio 0.43. Off by default.
#
# The first two are not equivalent, because sigma_data is pinned at 0.5 in both and so does not
# rescale with the data. scale=None -> derive so D_- matches theirs; sigma_max=None -> derive to
# match their ratio. Both are resolved in the data cell below, where D_- is known.
GEOM_SPECS = {
    'rescaled': dict(scale=None, sigma_max=80.0,
                     note='data scaled so D_- = 18.44; sigma_max literal'),
    'sigmamax': dict(scale=1.0,  sigma_max=None,
                     note='data as generated; sigma_max raised to match sigma_max/D_-'),
    'unit':     dict(scale=1.0,  sigma_max=80.0,
                     note='literal control: data as generated, sigma_max literal'),
}
GEOM_CHOICES = tuple(GEOM_SPECS)

# Default run = the two ways of matching their regime, both in one execution.
CFG['geometries'] = ['rescaled', 'sigmamax']

if SMOKE:
    CFG.update(epochs=300, eval_every=100, n_eval_samples=16, num_steps=10,
               model_channels_sweep=[4], eval_batch=16, save_resume_state=False)

# Environment overrides, so one SLURM array task can own one (geometry, arm) pair:
#     FIELD_GEOM=sigmamax FIELD_ARMS=32 sbatch ...
# Both accept space- or comma-separated lists.
_geom_env = os.environ.get('FIELD_GEOM')
if _geom_env:
    CFG['geometries'] = _geom_env.replace(',', ' ').split()
    print(f"FIELD_GEOM override -> {CFG['geometries']}")
for _g in CFG['geometries']:
    assert _g in GEOM_CHOICES, f'unknown geometry {_g!r}; choose from {GEOM_CHOICES}'

_arms_env = os.environ.get('FIELD_ARMS')
if _arms_env:
    CFG['model_channels_sweep'] = [int(c) for c in _arms_env.replace(',', ' ').split()]
    print(f"FIELD_ARMS override -> {CFG['model_channels_sweep']}")

# One file per (geometry, arm), never a shared one: concurrent array tasks would otherwise
# clobber each other. The plotting cells glob whatever is present.
RESULT_DIR = os.path.join(results_dir, 'baptista_matern_n2')
os.makedirs(RESULT_DIR, exist_ok=True)
STATE_DIR = RESULT_DIR

def arm_result_path(C, geom):
    return os.path.join(RESULT_DIR, f'arm_{geom}_c{C}_result.pt')

def arm_state_path(C, geom):
    return os.path.join(STATE_DIR, f'arm_{geom}_c{C}.pt')

def load_runs(geom):
    '''Collect every completed arm on disk for one geometry. Safe in a fresh kernel.'''
    out = {}
    for C in PAPER_FIG18_PARAMS:
        p = arm_result_path(C, geom)
        if os.path.exists(p):
            out[C] = torch.load(p, map_location='cpu', weights_only=False)
    return out

n_evals = CFG['epochs'] // CFG['eval_every']
n_arms = len(CFG['geometries']) * len(CFG['model_channels_sweep'])
print(f"{'SMOKE RUN' if SMOKE else 'FULL RUN'}")
print(f"  geometries    : {CFG['geometries']}")
for _g in CFG['geometries']:
    print(f"                  {_g:<9} -- {GEOM_SPECS[_g]['note']}")
print(f"  arms          : {CFG['model_channels_sweep']}  "
      f"({[f'{PAPER_FIG18_PARAMS[c]:,}' for c in CFG['model_channels_sweep']]} params)")
print(f"  total arms    : {n_arms}  (geometries x model sizes)")
print(f"  epochs        : {CFG['epochs']:,} each  (batch_mode={CFG['batch_mode']}, "
      f"{CFG['epochs'] * (N_TRAIN if CFG['batch_mode'] == 'main_py' else 1):,} optimizer updates)")
print(f"  evaluations   : {n_evals} x {CFG['n_eval_samples']} samples at {CFG['num_steps']} sampler steps")
print(f"  sampler       : {'ODE (deterministic Heun)' if CFG['S_churn'] == 0 else 'SDE (S_churn=%g)' % CFG['S_churn']}"
      f", sigma_min={CFG['sigma_min']}, rho={CFG['rho']}  (sigma_max is per-geometry)")
print(f"  results dir   : {RESULT_DIR}")
print(f"  one file per arm: arm_<geom>_c<C>_result.pt (+ arm_<geom>_c<C>.pt resume state)")


## The training data

The same two fields every other `n_train=2` result in this repo uses: the first two draws of the
standard 200-field multiband pool (`seed=42`, `normalize=True`, four Matérn components at length
scales 2 / 6 / 12 / 24 with weights 1.0 / 0.8 / 0.8 / 1.2). Nothing about the pool is changed —
only, on the `rescaled` arm, an overall multiplicative constant.

The reference geometry is their rectangles: two disjoint binary squares of $12^2$ and $14^2$
pixels, so $\|x_0\| = 12$, $\|x_1\| = 14$ and $D_- = \sqrt{144+196} = 18.439$ exactly.

In [ ]:
from multiband_data_utils import generate_multiband_dataset_postmask

# The standard pool -- identical call to every other field notebook in this folder.
components = [
    {"name": "coarse", "length_scale": 2.0,  "s": 2.0, "sigma_sq": 1.0, "band": (0.5, 4.0)},
    {"name": "mid1",   "length_scale": 6.0,  "s": 2.0, "sigma_sq": 1.0, "band": (4.0, 10.0)},
    {"name": "mid2",   "length_scale": 12.0, "s": 2.0, "sigma_sq": 1.0, "band": (10.0, 18.0)},
    {"name": "fine",   "length_scale": 24.0, "s": 2.0, "sigma_sq": 1.0, "band": (18.0, 32.0)},
]
_pool = generate_multiband_dataset_postmask(
    num_samples=200, grid_size=GRID, components=components,
    weights=[1.0, 0.8, 0.8, 1.2], seed=42, normalize=True,
)
_base = _pool['combined'][:N_TRAIN].reshape(N_TRAIN, 1, GRID, GRID).clone().float()


def _pair_dist(x):
    f = x.reshape(x.shape[0], -1)
    return torch.cdist(f, f)[0, 1].item()


# Their reference geometry: two disjoint binary squares of 12^2 and 14^2 pixels, so
# ||x_0|| = 12, ||x_1|| = 14 and D_- = sqrt(144 + 196) = 18.439 exactly.
RECT_D_MINUS   = math.sqrt(340.0)
RECT_NORMS     = (12.0, 14.0)
RECT_STD       = 0.19947                        # data.std() of their pair, for reference
RECT_SIGMA_MAX = 80.0                           # generate.py:27 default, used by main.py:134
RECT_RATIO     = RECT_SIGMA_MAX / RECT_D_MINUS  # 4.33865

D_UNIT = _pair_dist(_base)                      # the pair exactly as generated


def build_data(geom):
    '''Resolve one geometry name into its data, D_-, scale and sigma_max. Mutates nothing.'''
    spec = GEOM_SPECS[geom]
    scale = spec['scale'] if spec['scale'] is not None else RECT_D_MINUS / D_UNIT
    x = _base * scale
    D = _pair_dist(x)
    sigma_max = spec['sigma_max'] if spec['sigma_max'] is not None else RECT_RATIO * D
    return dict(geometry=geom, data=x, D_minus=D, scale=scale,
                sigma_max=sigma_max, ratio=sigma_max / D, note=spec['note'])


GEO = {g: build_data(g) for g in CFG['geometries']}

# Convenience alias for the sanity checks below. The run loop always uses GEO[...]['data'].
data = GEO[CFG['geometries'][0]]['data']

print(f"reference (their rectangles): ||x|| {RECT_NORMS[0]}/{RECT_NORMS[1]}, "
      f"D_- {RECT_D_MINUS:.4f}, sigma_max {RECT_SIGMA_MAX:g}, ratio {RECT_RATIO:.4f}, "
      f"std {RECT_STD:.5f}, mean 0.04150")
print(f"our pair as generated       : {GRID}x{GRID}, d = {GRID*GRID}, D_- {D_UNIT:.4f}\n")
print(f"{'geometry':>10} {'scale':>9} {'||x0||':>9} {'||x1||':>9} {'D_-':>10} "
      f"{'sigma_max':>10} {'smax/D_-':>9} {'std':>8}")
print('-' * 84)
for _g in GEOM_CHOICES:
    _gd = GEO[_g] if _g in GEO else build_data(_g)
    _n = _gd['data'].reshape(N_TRAIN, -1).norm(dim=1)
    _mark = '  <-' if _g in CFG['geometries'] else ''
    print(f"{_g:>10} {_gd['scale']:>9.5f} {_n[0]:>9.3f} {_n[1]:>9.3f} {_gd['D_minus']:>10.4f} "
          f"{_gd['sigma_max']:>10.4g} {_gd['ratio']:>9.4f} {_gd['data'].std():>8.5f}{_mark}")
print('-' * 84)
print("'<-' marks the geometries this run will train. sigma_data is held at "
      f"{CFG['sigma_data']} in all of them")
print('(networks.py:640 default, exactly as main.py leaves it), so the arms are NOT related')
print('by a simple rescaling of the dynamics -- that is the point of running both.')

# Hard guards: these are the numbers the geometry arms are defined by.
for _g, _gd in GEO.items():
    if _g == 'rescaled':
        assert abs(_gd['D_minus'] - RECT_D_MINUS) < 1e-3, _gd['D_minus']
        assert abs(_gd['ratio'] - RECT_RATIO) < 1e-3, _gd['ratio']   # float32 round-trip
    elif _g == 'sigmamax':
        assert abs(_gd['scale'] - 1.0) < 1e-12
        assert abs(_gd['D_minus'] - 185.8759) < 1e-2, _gd['D_minus']
        assert abs(_gd['ratio'] - RECT_RATIO) < 1e-3, _gd['ratio']   # float32 round-trip
    elif _g == 'unit':
        assert abs(_gd['scale'] - 1.0) < 1e-12
        assert abs(_gd['D_minus'] - 185.8759) < 1e-2, _gd['D_minus']

# Every arm is the same field pair up to an overall constant, so one figure covers them all;
# the title carries the geometry that produced it.
_show = GEO[CFG['geometries'][0]]
_v = float(_show['data'].abs().max())
_norms = _show['data'].reshape(N_TRAIN, -1).norm(dim=1)
fig, axes = plt.subplots(1, N_TRAIN, figsize=(6.4, 3.2))
for j in range(N_TRAIN):
    im = axes[j].imshow(_show['data'][j, 0], vmin=-_v, vmax=_v, cmap='RdBu_r')
    axes[j].set_xticks([]); axes[j].set_yticks([])
    axes[j].set_title(f'$x_{j}$,  $\\|x_{j}\\|$ = {_norms[j]:.2f}', fontsize=10)
fig.colorbar(im, ax=axes, fraction=0.035)
fig.suptitle(f'Training data — Matern pair, geometry={_show["geometry"]}, '
             f'$D_-$ = {_show["D_minus"]:.3f}', y=1.02)
plt.savefig(os.path.join(fig_dir, f'baptista_matern_n2_{_show["geometry"]}_training_data.png'),
            dpi=150, bbox_inches='tight')
plt.show()


## The metric

Their Figure-18 number — binarize at 0.5, count samples at Euclidean distance **exactly** 0 —
needs binary data. It does not transfer.

What does transfer is the metric their released code actually computes. `main.py:150-159`:

```python
L2_error[k] = torch.sum((samples_rand[j] - data[k])**2)   # squared L2, unnormalized
L2_dist[j]  = torch.min(L2_error)                         # to the nearest training image
L2_manifold_loss.append(torch.max(L2_dist))               # max over the eval batch
L2_mean_manifold_loss.append(torch.mean(L2_dist))         # mean over the eval batch
```

`post_process.py:29` plots this as *"$L^2$ distance to data manifold"*, the y-axis of Figure 17.
It is continuous and scale-carrying, so it works on fields verbatim. **This is the primary
readout**, logged as `l2_max` and `l2_mean` exactly as they log it.

Two secondary readouts, because a raw squared distance is hard to read across geometry arms:

* `nn_rel` — $\min_k \|x_{gen} - x_k\| \,/\, \mathrm{mean}_k \|x_k\|$, the convention
  `pixel_nn_stats` already uses in `edm_unet_memorization_transition.ipynb`, so the curve can be
  read against the existing field runs. Scale-free, so the two geometry arms are comparable.
* `fraction` — the share of samples with `nn_rel < 0.3`. **Threshold-dependent**, unlike their
  exact-match fraction, which is not. Report it as a convenience, never as the headline.

In [ ]:
@torch.no_grad()
def field_stats(x_gen, x_train, rel_threshold=0.3):
    '''main.py:150-159's L2-to-data-manifold, plus a scale-free relative version.

    x_gen:   (G, 1, N, N) generated fields
    x_train: (K, 1, N, N) training fields
    '''
    g = x_gen.detach().float().cpu().reshape(x_gen.shape[0], -1)
    t = x_train.detach().float().cpu().reshape(x_train.shape[0], -1)
    d2 = torch.cdist(g, t) ** 2                 # (G, K) squared L2 -- main.py:153
    l2_min = d2.min(dim=1).values               # main.py:156  L2_dist[j]
    nn_idx = d2.argmin(dim=1)
    nn_rel = l2_min.clamp_min(0).sqrt() / t.norm(dim=1).mean()
    return {
        'l2_max':        l2_min.max().item(),      # main.py:158  L2_manifold_loss
        'l2_mean':       l2_min.mean().item(),     # main.py:159  L2_mean_manifold_loss
        'l2_min':        l2_min.min().item(),
        'nn_rel_median': nn_rel.median().item(),
        'nn_rel_min':    nn_rel.min().item(),
        'fraction':      (nn_rel < rel_threshold).float().mean().item(),
        'nn_index':      nn_idx,
    }

# sanity: the training fields score 0 distance against themselves; matched-variance noise does not
_self = field_stats(data, data)
assert _self['l2_max'] < 1e-8 and _self['fraction'] == 1.0, _self
torch.manual_seed(0)
_noise = field_stats(torch.randn(64, 1, GRID, GRID) * data.std(), data)
assert _noise['fraction'] == 0.0, _noise
print(f"metric check: training data -> nn_rel {_self['nn_rel_median']:.2e}, fraction {_self['fraction']:.2f}")
print(f"              matched noise -> nn_rel {_noise['nn_rel_median']:.3f}, fraction {_noise['fraction']:.2f}")

## The gate — can this sampler reach the memorizing solution at all?

The rectangles notebook gates on `collapse_stats(data, data)['fraction'] == 1.0`: a trivial check,
because there the memorized output *is* a training image bit-for-bit.

For fields the equivalent question is sharper and worth asking before spending any GPU time.
The exact minimizer of the empirical denoising loss is the closed-form empirical-Bayes denoiser
(Baptista et al. Theorem 3.2),

$$D^\star(x;\sigma) \;=\; \sum_k w_k\, y_k, \qquad
  w_k \;\propto\; \exp\!\left(-\|x-y_k\|^2 / 2\sigma^2\right),$$

which is *by construction* fully memorizing. Pushing it through the same `edm_sampler` at the
arm's $\sigma_{max}$ asks: **if the network found the memorizing solution, would this sampler
produce collapsed samples?**

On the `unit` arm this is a live question, not a formality — at $\sigma_{max}/D_- = 0.43$ the
sampler starts below the mode-mixing scale. If the gate fails there, then a U-Net that does not
memorize on that arm tells us nothing about U-Nets, only about the sampler's noise range. The
gate result is recorded with every arm so the training curves are never read without it.

It is deliberately **not** an assert: a failing gate is a finding, and aborting the SLURM task
would throw away the training run that demonstrates it.

In [ ]:
class GMMDenoiser(torch.nn.Module):
    '''Exact empirical-Bayes denoiser for the N-point empirical measure = full memorization.

    Duck-types the EDMPrecond interface that edm_sampler needs: forward(x, sigma, class_labels),
    plus .sigma_min / .sigma_max / .round_sigma.
    '''
    sigma_min = 0.0
    sigma_max = float('inf')

    def __init__(self, y):
        super().__init__()
        self.register_buffer('y', y.reshape(y.shape[0], -1))

    def round_sigma(self, sigma):
        return torch.as_tensor(sigma)

    def forward(self, x, sigma, class_labels=None):
        shp = x.shape
        xf = x.reshape(shp[0], -1).to(self.y.dtype)
        d2 = torch.cdist(xf, self.y) ** 2                                    # (G, K)
        s = torch.as_tensor(sigma, dtype=self.y.dtype, device=xf.device).reshape(-1, 1)
        w = torch.softmax(-d2 / (2 * s ** 2), dim=1)
        return (w @ self.y).reshape(shp)



@torch.no_grad()
def run_gate(geo, cfg):
    '''Push the closed-form memorizing denoiser through this geometry's sampler.'''
    g = torch.Generator().manual_seed(cfg['latent_seed'])
    lat = torch.randn(cfg['n_sample_grid'], 1, GRID, GRID, generator=g).to(DEVICE)
    gmm = GMMDenoiser(geo['data'].to(DEVICE).to(SAMPLER_DTYPE))
    xg = edm_sampler(gmm, lat, num_steps=cfg['num_steps'],
                     sigma_min=cfg['sigma_min'], sigma_max=geo['sigma_max'], rho=cfg['rho'],
                     S_churn=cfg['S_churn'], S_min=cfg['S_min'], S_max=cfg['S_max'],
                     S_noise=cfg['S_noise'])
    st = field_stats(xg.float().cpu(), geo['data'], rel_threshold=cfg['rel_threshold'])
    st.pop('nn_index')
    st.update(geometry=geo['geometry'], sigma_max=geo['sigma_max'],
              sigma_max_over_D=geo['ratio'], D_minus=geo['D_minus'])
    st['passed'] = st['nn_rel_median'] < 0.05
    return st


GATES = {}
for _g in CFG['geometries']:
    GATES[_g] = run_gate(GEO[_g], CFG)
    _s = GATES[_g]
    print(f"closed-form memorizing denoiser through the same sampler -- geometry={_g}, "
          f"sigma_max={_s['sigma_max']:.4g}, sigma_max/D_- = {_s['sigma_max_over_D']:.3f}:")
    print(f"  L2 to manifold  max {_s['l2_max']:.4e}   mean {_s['l2_mean']:.4e}")
    print(f"  nn_rel median   {_s['nn_rel_median']:.5f}")
    print(f"  fraction < {CFG['rel_threshold']}   {_s['fraction']:.2f}")
    if _s['passed']:
        print('  GATE PASSED -- the memorizing solution is reachable by this sampler here.')
        print('                 A U-Net that does not collapse is a statement about the U-Net.\n')
    else:
        print('  GATE FAILED -- even the exact memorizing score does not produce collapsed')
        print('                 samples at this sigma_max. Any non-collapse below is a statement')
        print('                 about the sampler noise range, NOT the network or the data.\n')


## Training

`main.py:73-107`, transcribed. Two changes from the rectangles notebook, both mechanical: the
eval latents are drawn at `GRID` rather than a hardcoded 64, and `collapse_stats` is replaced by
`field_stats`. The optimizer loop, lr ramp-up, gradient guard, EMA update and resume state are
byte-identical.

Resume state (net + EMA + optimizer + **RNG**) is written at every evaluation, so a SLURM timeout
resumes bit-identically rather than restarting: the loader's shuffle order, the `EDMLoss` sigma
draws and its noise all come from the restored generator.

In [ ]:
@torch.no_grad()
def generate_samples(ema_net, n_samples, cfg, device, seed, sigma_max):
    '''Draw n_samples with the EDM sampler, chunked to fit in memory. main.py samples from EMA.'''
    ema_net.eval()
    out = []
    remaining = n_samples
    chunk_i = 0
    while remaining > 0:
        b = min(cfg['eval_batch'], remaining)
        g = torch.Generator().manual_seed(seed + 1000 * chunk_i)
        latents = torch.randn(b, 1, GRID, GRID, generator=g).to(device)
        x = edm_sampler(ema_net, latents,
                        num_steps=cfg['num_steps'], sigma_min=cfg['sigma_min'],
                        sigma_max=sigma_max, rho=cfg['rho'], S_churn=cfg['S_churn'],
                        S_min=cfg['S_min'], S_max=cfg['S_max'], S_noise=cfg['S_noise'])
        out.append(x.float().cpu())
        remaining -= b
        chunk_i += 1
    return torch.cat(out, dim=0)


def run_arm(model_channels, geo, cfg, device, resume=True, log=print):
    '''One model size, transcribing main.py's training loop. Returns the evaluation log.'''
    state_path = arm_state_path(model_channels, geo['geometry'])

    torch.manual_seed(cfg['seed'])
    net = build_net(model_channels, device)
    net.train().requires_grad_(True)
    ema = copy.deepcopy(net).eval().requires_grad_(False)
    optimizer = torch.optim.Adam(net.parameters(), lr=cfg['lr'],
                                 betas=list(cfg['betas']), eps=cfg['eps'])
    loss_fn = EDMLoss(P_mean=cfg['P_mean'], P_std=cfg['P_std'], sigma_data=cfg['sigma_data'])

    cur_nimg, opt_step, start_epoch = 1, 0, 0     # main.py:18 starts cur_nimg at 1
    eval_log, loss_hist = [], []

    if resume and os.path.exists(state_path):
        # to CPU: load_state_dict re-places model/optimizer tensors onto the params'
        # device by itself, and the saved RNG state must stay a CPU ByteTensor.
        st = torch.load(state_path, map_location='cpu', weights_only=False)
        net.load_state_dict(st['net']); ema.load_state_dict(st['ema'])
        optimizer.load_state_dict(st['opt'])
        cur_nimg, opt_step, start_epoch = st['cur_nimg'], st['opt_step'], st['epoch']
        eval_log, loss_hist = st['eval_log'], st['loss_hist']
        # Restore the RNG too, so a resumed run is bit-identical to an uninterrupted one.
        torch.set_rng_state(st['rng_cpu'])
        if st.get('rng_cuda') is not None and torch.cuda.is_available():
            torch.cuda.set_rng_state_all(st['rng_cuda'])
        if st.get('rng_mps') is not None and torch.backends.mps.is_available():
            torch.mps.set_rng_state(st['rng_mps'])
        log(f'  resumed from epoch {start_epoch:,}')

    data = geo['data']          # this geometry's pair; run_arm never reads the global
    x_train_cpu = data
    if cfg['batch_mode'] == 'main_py':
        loader = torch.utils.data.DataLoader(
            torch.utils.data.TensorDataset(data), batch_size=1, shuffle=True)   # main.py:37
        def epoch_batches():
            for (x,) in loader:
                yield x
    elif cfg['batch_mode'] == 'paper_bs2':
        def epoch_batches():
            yield data[torch.randperm(N_TRAIN)]
    else:
        raise ValueError(cfg['batch_mode'])

    t_start = time.time()
    for ep in range(start_epoch, cfg['epochs']):
        err, count = 0.0, 0
        net.train()
        for x in epoch_batches():
            optimizer.zero_grad(set_to_none=True)
            x = x.to(device)
            loss = loss_fn(net, x).sum() / x.size(0)          # main.py:84
            loss.backward()

            for g in optimizer.param_groups:                   # main.py:88-89
                g['lr'] = cfg['lr'] * min(cur_nimg / max(cfg['lr_rampup_kimg'] * 1000, 1e-8), 1)
            for param in net.parameters():                     # main.py:91-93
                if param.grad is not None:
                    torch.nan_to_num(param.grad, nan=0, posinf=1e5, neginf=-1e5, out=param.grad)
            optimizer.step()

            ema_halflife_nimg = cfg['ema_halflife_kimg'] * 1000        # main.py:96-102
            if cfg['ema_rampup_ratio'] is not None:
                ema_halflife_nimg = min(ema_halflife_nimg, cur_nimg * cfg['ema_rampup_ratio'])
            ema_beta = 0.5 ** (x.size(0) / max(ema_halflife_nimg, 1e-8))
            for p_ema, p_net in zip(ema.parameters(), net.parameters()):
                p_ema.copy_(p_net.detach().lerp(p_ema, ema_beta))

            err += loss.item()
            cur_nimg += x.size(0)
            count += x.size(0)
            opt_step += 1

        loss_hist.append(err / count)

        if (ep + 1) % cfg['eval_every'] == 0:
            n_done = (ep + 1) // cfg['eval_every']
            x_gen = generate_samples(ema, cfg['n_eval_samples'], cfg, device,
                                     seed=cfg['latent_seed'] + n_done,
                                     sigma_max=geo['sigma_max'])
            st = field_stats(x_gen, x_train_cpu, rel_threshold=cfg['rel_threshold'])
            row = dict(epoch=ep + 1, opt_step=opt_step, cur_nimg=cur_nimg,
                       lr=optimizer.param_groups[0]['lr'],
                       loss=float(np.mean(loss_hist[-cfg['eval_every']:])),
                       l2_max=st['l2_max'], l2_mean=st['l2_mean'], l2_min=st['l2_min'],
                       nn_rel_median=st['nn_rel_median'], nn_rel_min=st['nn_rel_min'],
                       fraction=st['fraction'],
                       samples=x_gen[:cfg['n_sample_grid']].clone())
            eval_log.append(row)

            el = time.time() - t_start
            frac_done = (ep + 1 - start_epoch) / max(cfg['epochs'] - start_epoch, 1)
            eta = el / max(frac_done, 1e-9) - el
            log(f"  epoch {ep+1:>7,} | step {opt_step:>7,} | lr {row['lr']:.2e} | "
                f"loss {row['loss']:.4f} | L2(mean) {st['l2_mean']:.4e} | "
                f"nn_rel(med) {st['nn_rel_median']:.4f} | frac {st['fraction']:.2f} | "
                f"{el/60:.1f}m elapsed, ~{eta/60:.0f}m left")

            if cfg['save_resume_state']:
                torch.save(dict(net=net.state_dict(), ema=ema.state_dict(),
                                opt=optimizer.state_dict(), cur_nimg=cur_nimg,
                                opt_step=opt_step, epoch=ep + 1,
                                eval_log=eval_log, loss_hist=loss_hist,
                                rng_cpu=torch.get_rng_state(),
                                rng_cuda=(torch.cuda.get_rng_state_all()
                                          if torch.cuda.is_available() else None),
                                rng_mps=(torch.mps.get_rng_state()
                                         if torch.backends.mps.is_available() else None)),
                           state_path)

    return dict(eval_log=eval_log, loss_hist=loss_hist,
                n_params=count_params(net), model_channels=model_channels,
                geometry=geo['geometry'], sigma_max=geo['sigma_max'],
                minutes=(time.time() - t_start) / 60)

### Run the sweep

Idempotent: an arm whose result file already holds a full evaluation log is skipped, and a
partial run resumes from its state file. Safe to re-execute after a SLURM timeout.

In [ ]:
NOTE = ('Baptista et al. arXiv:2501.15785 section 5.4 config, applied to two multiband Matern '
        'fields instead of two binary rectangles. Config transcribed from '
        'baptistar/DiffusionModelDynamics RectangleImages/main.py; Figure 18 protocol from the '
        'paper text. Changed vs that repo: img_resolution 64->128 and attn_resolutions [16]->[32] '
        '(same level, parameter counts preserved exactly), the collapse metric (their exact-match '
        'metric needs binary data; replaced by their own L2-to-manifold from main.py:150-159), '
        'the data, and the geometry arm (data scale and/or sampler sigma_max -- see GEOM_SPECS). '
        'Sampler is deterministic Heun at S_churn=0 (the released default) unless CFG_SDE_ALT was '
        'applied. NOT comparable to any edm_unet_* result in this repo.')

n_expect = CFG['epochs'] // CFG['eval_every']
for geom in CFG['geometries']:
    geo, gate = GEO[geom], GATES[geom]
    print(f'######## geometry={geom}  sigma_max={geo["sigma_max"]:.4g}  '
          f'D_-={geo["D_minus"]:.3f}  ratio={geo["ratio"]:.3f}  '
          f'gate={"PASSED" if gate["passed"] else "FAILED"} ########', flush=True)
    for C in sorted(CFG['model_channels_sweep']):
        p = arm_result_path(C, geom)
        if os.path.exists(p):
            prev = torch.load(p, map_location='cpu', weights_only=False)
            if len(prev['eval_log']) >= n_expect:
                print(f'=== {geom} c{C} already complete, skipping ===', flush=True)
                continue
        print(f'=== geometry={geom}  model_channels={C}  '
              f'({PAPER_FIG18_PARAMS[C]:,} params) ===', flush=True)
        res = run_arm(C, geo, CFG, DEVICE, resume=True)
        res.update(cfg={k: v for k, v in CFG.items()}, data=geo['data'], note=NOTE,
                   gmm_gate=gate, D_minus=geo['D_minus'], scale=geo['scale'],
                   sigma_max=geo['sigma_max'], geom_spec=GEOM_SPECS[geom])
        torch.save(res, p)
        print(f"  done in {res['minutes']:.1f} min -> {p}", flush=True)

for geom in CFG['geometries']:
    _r = load_runs(geom)
    print(f'\narms on disk for geometry={geom}: '
          + (', '.join('C%d=%s params' % (C, format(_r[C]['n_params'], ',')) for C in sorted(_r))
             or 'none'))


## Result

Both geometry arms are plotted together where they exist on disk, so a single kernel after the
array job shows the whole picture. The dashed horizontal line is the closed-form memorizing
denoiser through the same sampler — the gate. A training curve that does not reach it has not
memorized; a *gate* that does not reach zero means the arm could never have shown memorization
in the first place.

In [ ]:
all_runs = {g: load_runs(g) for g in GEOM_CHOICES}
present = [g for g in GEOM_CHOICES if all_runs[g]]
print('geometries on disk:', present or 'none')

if present:
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
    cmap = plt.get_cmap('viridis')
    styles = {'rescaled': '-', 'sigmamax': '--', 'unit': ':'}

    for g in present:
        runs_g = all_runs[g]
        order = sorted(runs_g)
        for i, C in enumerate(order):
            r = runs_g[C]
            ep = [e['epoch'] for e in r['eval_log']]
            col = cmap(i / max(len(order) - 1, 1))
            lab = f"{g}, {r['n_params']:,}"
            axes[0].plot(ep, [e['nn_rel_median'] for e in r['eval_log']],
                         color=col, ls=styles[g], lw=1.5, label=lab)
            axes[1].plot(ep, [e['l2_mean'] for e in r['eval_log']],
                         color=col, ls=styles[g], lw=1.5, label=lab)
            axes[2].plot(ep, [e['fraction'] for e in r['eval_log']],
                         color=col, ls=styles[g], lw=1.5, label=lab)
        gate = runs_g[order[0]].get('gmm_gate')
        if gate:
            axes[0].axhline(gate['nn_rel_median'], color='k', ls=':', lw=1.0)
            axes[1].axhline(max(gate['l2_mean'], 1e-12), color='k', ls=':', lw=1.0)

    axes[0].set_yscale('log')
    axes[0].set_ylabel('median $nn_{rel}$ to nearest training field')
    axes[0].set_title('relative distance (scale-free)\ndotted = closed-form memorizing denoiser')
    axes[1].set_yscale('log')
    axes[1].set_ylabel('mean $L^2$ distance to data manifold')
    axes[1].set_title("main.py:159's metric (Figure 17 y-axis)")
    axes[2].set_ylabel(f"fraction with $nn_{{rel}}$ < {CFG['rel_threshold']}")
    axes[2].set_ylim(-0.02, 1.02)
    axes[2].set_title('collapse fraction (threshold-dependent)')
    for ax in axes:
        ax.set_xlabel('Epochs (2 optimization steps each)')
        ax.legend(fontsize=7)
    fig.suptitle('Baptista et al. §5.4 config on Matern fields, $N=2$', y=1.03)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, 'baptista_matern_n2_curves.png'), dpi=150,
                bbox_inches='tight')
    plt.show()

In [ ]:
# The gate, and the ordering claim -- per geometry.
for g in present:
    runs_g = all_runs[g]
    order = sorted(runs_g)
    gate = runs_g[order[0]].get('gmm_gate', {})
    print(f"=== geometry = {g}   sigma_max = {gate.get('sigma_max', float('nan')):.4g}   sigma_max/D_- = {gate.get('sigma_max_over_D', float('nan')):.3f} ===")
    print(f"    closed-form memorizing denoiser: nn_rel(med) {gate.get('nn_rel_median', float('nan')):.5f}"
          f"  -> gate {'PASSED' if gate.get('nn_rel_median', 1) < 0.05 else 'FAILED'}")
    print(f"    {'params':>12} {'model_ch':>9} {'min nn_rel':>11} {'final nn_rel':>13} "
          f"{'peak frac':>10} {'first ep frac>=0.9':>19}")
    print('    ' + '-' * 78)
    onsets = {}
    for C in order:
        r = runs_g[C]
        nr = [e['nn_rel_median'] for e in r['eval_log']]
        fr = [e['fraction'] for e in r['eval_log']]
        ep = [e['epoch'] for e in r['eval_log']]
        hit = next((ep[i] for i, v in enumerate(fr) if v >= 0.9), None)
        onsets[C] = hit
        print(f"    {r['n_params']:>12,} {C:>9} {min(nr):>11.4f} {nr[-1]:>13.4f} "
              f"{max(fr):>10.2f} {(f'{hit:,}' if hit else 'not reached'):>19}")
    reached = [C for C in order if onsets[C] is not None]
    print(f"    GATE  : {len(reached)}/{len(order)} arms reach collapse fraction >= 0.9")
    if len(reached) >= 2:
        mono = all(onsets[reached[i]] >= onsets[reached[i + 1]] for i in range(len(reached) - 1))
        print(f"    ORDER : onset monotone decreasing in model size: {mono}")
    print()

if len(present) >= 2:
    print('Reminder: these arms differ in data scale and/or sampler sigma_max. sigma_data is '
          'held at 0.5 in all of them and P_mean/P_std fix the training sigma range in '
          'absolute terms, so they are NOT related by a simple rescaling of the dynamics. '
          'Compare them to each other, not to any edm_unet_* number.')

### Samples vs training time

The Figure-17 analogue. Rows are evaluation checkpoints, columns are generated fields; the
training pair is repeated in the last row for reference. Colour scale is shared and fixed to the
training data's range, so a sample that has collapsed onto a training field looks identical to it.

In [ ]:
for g in present:
    runs_g = all_runs[g]
    C_show = max(runs_g)
    r = runs_g[C_show]
    log_rows = r['eval_log']
    if not log_rows:
        continue
    want = [2000, 4000, 6000, 10000, 20000, 50000]
    avail = [e['epoch'] for e in log_rows]
    picks, seen = [], set()
    for w in want:
        j = int(np.argmin([abs(a - w) for a in avail]))
        if j not in seen:
            seen.add(j); picks.append(j)

    n_show = min(6, log_rows[0]['samples'].shape[0])
    d_show = r.get('data', data)
    vmax = float(d_show.abs().max())
    fig, axes = plt.subplots(len(picks) + 1, n_show, squeeze=False,
                             figsize=(1.1 * n_show, 1.15 * (len(picks) + 1)))
    for row, j in enumerate(picks):
        s = log_rows[j]['samples']
        for k in range(n_show):
            axes[row][k].imshow(s[k, 0], vmin=-vmax, vmax=vmax, cmap='RdBu_r')
            axes[row][k].set_xticks([]); axes[row][k].set_yticks([])
        axes[row][0].set_ylabel(f"{log_rows[j]['epoch']//1000}k\n"
                                f"{log_rows[j]['nn_rel_median']:.3f}",
                                fontsize=7, rotation=0, ha='right', va='center')
    for k in range(n_show):
        ax = axes[-1][k]
        ax.set_xticks([]); ax.set_yticks([])
        if k < d_show.shape[0]:
            ax.imshow(d_show[k, 0], vmin=-vmax, vmax=vmax, cmap='RdBu_r')
        else:
            ax.axis('off')
    axes[-1][0].set_ylabel('train', fontsize=7, rotation=0, ha='right', va='center')
    fig.suptitle(f'Samples vs training time — geometry={g}, model_channels={C_show} '
                 f'({r["n_params"]:,} params)\nrow label: epochs / median $nn_{{rel}}$',
                 fontsize=10)
    plt.tight_layout()
    plt.savefig(os.path.join(fig_dir, f'baptista_matern_n2_{g}_samples.png'), dpi=150,
                bbox_inches='tight')
    plt.show()

## Notes

**What a result here does and does not mean.**

* If the fields memorize under this config, the rectangles/fields distinction is not the driver,
  and the flat $n_{train}=2$ curve in `edm_unet_memorization_transition.ipynb` is explained by
  that notebook's $\sigma_{max}$, update budget and sampler class rather than by the data.
* If the fields do **not** memorize where the rectangles do, **and the gate passed**, then the
  memorizing solution was reachable and the network did not find it — a data-dependence result.
* If the gate **failed** on an arm, that arm says nothing about networks or data. It says the
  sampler's noise range never entered the regime where the two modes are confusable. Report it
  as ruled-out, not as diagnosed.

**Differences from the rectangles run that shift timing and must not be read as data effects.**

1. `loss.sum()/B` sums over $d = 16384$ pixels rather than 4096, so the gradient is $4\times$
   larger at the same learning rate.
2. The attention bottleneck operates on $32^2 = 1024$ tokens rather than $16^2 = 256$.
3. The lr ramp-up (`lr_rampup_kimg=10000`, `cur_nimg` advancing by 1 per update) caps the
   effective learning rate at $\sim10^{-5}$ over this budget, exactly as in their run.

**Known discrepancies in the upstream material**, carried over unchanged and flagged in the
rectangles notebook: `main.py:16` defines `bsize = 2` and never uses it (line 37 builds the
loader at `batch_size=1`); and `main.py:134` calls `edm_sampler` without `S_churn`, whose default
is 0, giving the deterministic ODE where §5.4 describes an SDE. `CFG['batch_mode']` and
`CFG_SDE_ALT` expose both readings.